# OLS — Linear Regression (Turkey, Pipeline B)

**Model**: `sklearn.linear_model.LinearRegression`  
**Target**: `ngdprsaxdctrq` (quarterly log-diff real GDP, Turkey)  
**Variables**: Cat2 — ipi_sa, usd_try_avg, cpi_sa, fin_acc + 3 COVID = 7 total  
**Train**: 1995-Q2 to 2011-Q4 / **Val**: 2012-Q1 to 2017-Q4 / **Test**: 2018-Q1 to 2025-Q4 (32 quarters)  
**Scaling**: No / **HP tuning**: No / **n_lags**: 3


In [1]:
import sys, os
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

sys.path.insert(0, os.path.join(os.path.pardir, 'turkey_data'))
from turkey_helpers import (
    gen_lagged_data, make_supervised_vintage_frame, flatten_data, mean_fill_dataset,
    get_features, load_data,
    PREDICTIONS_DIR, TARGET, COVID
)

SEED = 42
np.random.seed(SEED)

TRAIN_START = '1995-01-01'
TRAIN_END   = '2011-12-31'
VAL_END     = '2017-12-31'
TEST_START  = '2018-01-01'
TEST_END    = '2025-12-31'
N_LAGS      = 3
VINTAGES    = {'m1': -2, 'm2': -1, 'm3': 0, 'post1': 1, 'post2': 2}

print('OLS (Turkey) — Cat2 variables, n_lags={}, scaling=NO'.format(N_LAGS))


OLS (Turkey) — Cat2 variables, n_lags=3, scaling=NO


In [2]:
monthly, _, metadata = load_data()
features = get_features('cat2', with_covid=True)
print('Features: {} -> {}'.format(len(features), features))
print('Target: {}'.format(TARGET))


Features: 7 -> ['ipi_sa', 'usd_try_avg', 'cpi_sa', 'fin_acc', 'covid_2020q2', 'covid_2020q3', 'covid_2020q4']
Target: ngdprsaxdctrq


In [3]:
test_quarters = monthly[
    (monthly['date'] >= TEST_START) &
    (monthly['date'] <= TEST_END) &
    monthly['date'].dt.month.isin([3, 6, 9, 12])
]['date'].tolist()

predictions  = {v: [] for v in VINTAGES}
actuals_list = []
failed = 0

for i, q_date in enumerate(test_quarters):
    pd_q = pd.Timestamp(q_date)
    actual = monthly[monthly['date'] == pd_q][TARGET].values[0]
    actuals_list.append(actual)

    for vintage_name, offset in VINTAGES.items():
        vintage_date = pd_q + pd.DateOffset(months=offset)
        train_flat, test_flat, _ = make_supervised_vintage_frame(
            metadata, monthly, TARGET, features, TRAIN_START, pd_q, vintage_date, N_LAGS
        )

        feature_cols = [
            c for c in train_flat.columns
            if c != 'date' and c != TARGET
            and any(c == f or c.startswith(f + '_') for f in features)
        ]

        if len(train_flat) < 10 or len(test_flat) != 1:
            predictions[vintage_name].append(np.nan)
            continue

        try:
            model = LinearRegression()
            model.fit(train_flat[feature_cols].values, train_flat[TARGET].values)
            pred = model.predict(test_flat[feature_cols].values)[0]
        except Exception:
            pred = np.nan
            failed += 1

        predictions[vintage_name].append(pred)

    if (i + 1) % 8 == 0:
        print('  {} / {} quarters'.format(i + 1, len(test_quarters)))

print('Done. {} quarters, {} failures.'.format(len(test_quarters), failed))


  8 / 32 quarters


  16 / 32 quarters


  24 / 32 quarters


  32 / 32 quarters
Done. 32 quarters, 0 failures.


In [4]:
os.makedirs(PREDICTIONS_DIR, exist_ok=True)
for vn in VINTAGES:
    df = pd.DataFrame({
        'date': test_quarters, 'actual': actuals_list,
        'prediction': predictions[vn],
    })
    path = os.path.join(PREDICTIONS_DIR, 'ols_{}.csv'.format(vn))
    df.to_csv(path, index=False)
    print('Saved {} ({} rows) pred=[{:.4f},{:.4f}]'.format(
        os.path.basename(path), len(df), df['prediction'].min(), df['prediction'].max()))


Saved ols_m1.csv (32 rows) pred=[-0.0041,0.0367]
Saved ols_m2.csv (32 rows) pred=[-0.0275,0.0735]
Saved ols_m3.csv (32 rows) pred=[-0.2156,0.1503]
Saved ols_post1.csv (32 rows) pred=[-0.1625,0.1590]
Saved ols_post2.csv (32 rows) pred=[-0.1062,0.1666]


In [5]:
def rmse(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.sqrt(np.mean((a[m]-p[m])**2)) if m.sum()>0 else np.nan

def mae(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.mean(np.abs(a[m]-p[m])) if m.sum()>0 else np.nan

panels = {
    'Pre-crisis (2018-2019)': ('2018-01-01', '2019-12-31'),
    'COVID      (2020-2021)': ('2020-04-01', '2021-12-31'),
    'Post-COVID (2022-2025)': ('2022-01-01', '2025-12-31'),
    'Full test  (2018-2025)': ('2018-01-01', '2025-12-31'),
}
a = np.array(actuals_list)
d = pd.to_datetime(test_quarters)
print('OLS (Turkey) — Evaluation by Panel & Vintage')
for pn, (ps, pe) in panels.items():
    m = (d >= ps) & (d <= pe)
    print('--- {} ---'.format(pn))
    for vn in VINTAGES:
        pp = np.array(predictions[vn])
        print('  {}  RMSFE={:.6f}  MAE={:.6f}  N={}'.format(
            vn, rmse(a[m], pp[m]), mae(a[m], pp[m]), m.sum()))


OLS (Turkey) — Evaluation by Panel & Vintage
--- Pre-crisis (2018-2019) ---
  m1  RMSFE=0.014381  MAE=0.011892  N=8
  m2  RMSFE=0.011021  MAE=0.010329  N=8
  m3  RMSFE=0.016736  MAE=0.011864  N=8
  post1  RMSFE=0.026970  MAE=0.019035  N=8
  post2  RMSFE=0.021392  MAE=0.016738  N=8
--- COVID      (2020-2021) ---
  m1  RMSFE=0.068649  MAE=0.042038  N=7
  m2  RMSFE=0.049227  MAE=0.035500  N=7
  m3  RMSFE=0.042979  MAE=0.024681  N=7
  post1  RMSFE=0.022632  MAE=0.015307  N=7
  post2  RMSFE=0.017855  MAE=0.014881  N=7
--- Post-COVID (2022-2025) ---
  m1  RMSFE=0.017241  MAE=0.014145  N=16
  m2  RMSFE=0.017653  MAE=0.013264  N=16
  m3  RMSFE=0.019542  MAE=0.015962  N=16
  post1  RMSFE=0.016542  MAE=0.012975  N=16
  post2  RMSFE=0.021612  MAE=0.018903  N=16
--- Full test  (2018-2025) ---
  m1  RMSFE=0.035131  MAE=0.019544  N=32
  m2  RMSFE=0.026959  MAE=0.017553  N=32
  m3  RMSFE=0.025921  MAE=0.016809  N=32
  post1  RMSFE=0.021324  MAE=0.015460  N=32
  post2  RMSFE=0.020463  MAE=0.017071  N=